### 데이터 분석 컨설팅
* 영도구청 신성장전략과
* 불법주정차 관리 필요지역과 공영주차장 건립 위치 선정분석

### 라이브러리와 데이터경로 변수설정

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from ydata_profiling import profile_report

pd.set_option("display.max_column", 100)

grid_path = "../../../../10_분석업무/05_구군요청/26200_영도구/2024년/20241106/신성장전략과/02_기초자료/분석용격자/"
data_path = "../../../../10_분석업무/05_구군요청/26200_영도구/2024년/20241106/신성장전략과/02_기초자료/제공데이터/"
publ_path = "../../../../10_분석업무/05_구군요청/26200_영도구/2024년/20241106/신성장전략과/02_기초자료/공공데이터/"
pupl_path = "../../../../10_분석업무/05_구군요청/26200_영도구/2024년/20241106/신성장전략과/02_기초자료/민간데이터/"
prfl_path = "../../../../10_분석업무/05_구군요청/26200_영도구/2024년/20241106/신성장전략과/02_기초자료/프로파일링/"
oupu_path = "../../../../10_분석업무/05_구군요청/26200_영도구/2024년/20241106/신성장전략과/04_분석결과/"

# 입지선정을 위한 기준격자 데이터
grid = gpd.read_file(grid_path + "nlsp_021001007_26200.shp", encoding="utf-8").to_crs(5179)
grid = grid[['gid', 'geometry']]

# 생활인구 격자데이터
grid50 = gpd.read_file(grid_path + "busan_50cell_v21_26200.shp").to_crs(5179)


### 제공데이터 프로파일링

In [2]:
print("불법주정차 과태료 부과 건수 데이터 프로파일링")
illegal1 = pd.read_excel(data_path + "불법주정차 과태료 부과 건수('22년 7월~'24년 7월).xlsx", sheet_name='22.8월~23.7월', engine="openpyxl")
illegal2 = pd.read_excel(data_path + "불법주정차 과태료 부과 건수('22년 7월~'24년 7월).xlsx", sheet_name='23.8월~24.7월', engine="openpyxl")
illegal = pd.concat([illegal1, illegal2], ignore_index=True)
print(illegal.shape)
#illegal.to_csv(data_path + "불법주정차 과태료 부과건수.csv", encoding="utf-8", index=False)
#print("불법주정차 단속위치의 좌표변환을 위한 병합작업 실시, 좌표추가 후 프로파일링 재 실시")
#illegal = pd.read_excel(data_path + "불법주정차 과태료 부과 건수_좌표추가.xlsx", engine="openpyxl")
#illegal_report = illegal.profile_report()
#illegal_report.to_file(prfl_path + "불법주정차 과태료 부과 건수 프로파일링.html")
print()
print("불법주정차 단속 CCTV 정보 프로파일링")
cctv = pd.read_excel(data_path + "불법주정차 단속 CCTV 정보_좌표추가.xlsx", engine="openpyxl")
cctv_report = cctv.profile_report()
#cctv_report.to_file(prfl_path + "불법주정차 단속 CCTV 정보 프로파일링.html")
print()
print("공영주차장 관리 현황 프로파일링")
plc_prk1 = pd.read_excel(data_path + "공영주차장 관리 현황(영도구)_좌표추가.xlsx", engine="openpyxl")
plc_prk1_report = plc_prk1.profile_report()
#plc_prk1_report.to_file(prfl_path + "공영주차장 관리 현황(영도구)_전처리 프로파일링.html")
print()
print("건축물대장 현황 프로파일링")
buld1 = pd.read_excel(data_path + "일반건축물대장 현황.xlsx", engine="openpyxl")
buld2 = pd.read_excel(data_path + "집합건축물대장 현황.xlsx", engine="openpyxl")
buld = pd.concat([buld1, buld2], ignore_index=True)
print(buld.shape)
#buld.to_csv(data_path + "건축물대장 현황.csv", encoding="utf-8", index=False)
print("대지주소의 좌표변환을 위한 병합작업 실시, 좌표추가 후 프로파일링 재 실시")
buld = pd.read_excel(data_path + "건축물대장_상업시설_현황_좌표추가.xlsx", engine="openpyxl")
buld_report = buld.profile_report()
#buld_report.to_file(oprfl_path + "건축물대장 현황 프로파일링.html")
print()
print("차량등록대장 프로파일링")
cars = pd.read_excel(data_path + "차량등록대장_좌표추가.xlsx", engine="openpyxl")
cars_report = cars.profile_report()
#cars_report.to_file(prfl_path + "차량등록대장 프로파일링.html")


불법주정차 과태료 부과 건수 데이터 프로파일링
(51627, 8)

불법주정차 단속 CCTV 정보 프로파일링

공영주차장 관리 현황 프로파일링

건축물대장 현황 프로파일링
(20199, 14)
대지주소의 좌표변환을 위한 병합작업 실시, 좌표추가 후 프로파일링 재 실시

차량등록대장 프로파일링


### 데이터 전처리
1. 불법주정차 관리지역과 공영주차장 건립위치 분석용 데이터 전처리
2. 공통 Feature 위주로 전처리 수행

In [127]:
print("불법주정차 과태료 부과건수에 공간속성 부여하기: 차량번호 기준 51,627개 중 14,939개 미부여, 36,688개 부여")
geom = gpd.points_from_xy(illegal.경도, illegal.위도, crs=4326)
illegal = gpd.GeoDataFrame(illegal, geometry=geom).to_crs(5179)
illegal = illegal.loc[(~illegal['위도'].isnull()) & (~illegal['경도'].isnull())]
print()
print("도로명주소 시군구 경계에서 영도구를 선택하고 도로구간을 추출")
scco_sig = gpd.read_file(publ_path + "TL_SCCO_SIG.shp", encoding="cp949").set_crs(5179)
scco_sig_26200 = scco_sig.loc[scco_sig['SIG_CD']=='26200']
sprd_mng = gpd.read_file(publ_path + "TL_SPRD_MANAGE.shp", encoding="cp949").set_crs(5179)
sprd_mng_26200 = gpd.clip(sprd_mng, scco_sig_26200, keep_geom_type=False)
print()
print("불법주정차 과태료 부과 건수에 도로타입 적용 → 도로명DB 도로구간의 광역도로구분코드 적용")
illegal_wdr = gpd.sjoin_nearest(illegal, sprd_mng[['geometry', 'WDR_RD_CD']], how="left", distance_col="dist_to_road")
illegal_wdr.rename(columns={'WDR_RD_CD': 'wdr_rd_cd'}, inplace=True)
print("불법주정차 과태료 부과 건수 → 불법주정차 발생을 의미하므로, 발생통계 집계가능, 이를 국토통계격자에 추가")
illegal_wdr['X'] = illegal_wdr.geometry.x.round(3)
illegal_wdr['Y'] = illegal_wdr.geometry.y.round(3)
illegal_cnt = illegal_wdr.groupby(['단속장소', 'X', 'Y', '최초과태료금액', 'wdr_rd_cd'])[['차량번호']].count().reset_index()
illegal_cnt.rename(columns={'최초과태료금액': 'fine_amount'}, inplace=True)
geom = gpd.points_from_xy(illegal_cnt['X'], illegal_cnt['Y'])
illegal_cnt = gpd.GeoDataFrame(illegal_cnt, geometry=geom, crs=5179)
illegal_cnt.columns = ['IeglAddr', 'X', 'Y', 'fine_amount', 'wdr_rd_cd', 'CtlCnt', 'geometry']
overlay1 = gpd.overlay(grid, illegal_cnt, "intersection", keep_geom_type=False)
counted1 = overlay1.groupby(['gid', 'wdr_rd_cd'])[['CtlCnt', 'fine_amount']].sum().reset_index()
mrg_gdf = grid.merge(counted1, on="gid", how="left")
mrg_gdf['CtlCnt'] = mrg_gdf['CtlCnt'].fillna(0)
mrg_gdf['CtlCnt'] = mrg_gdf['CtlCnt'].fillna(0)
mrg_gdf = mrg_gdf[['gid', 'CtlCnt', 'fine_amount', 'wdr_rd_cd']]
print()
print("기존 공영주차장 수용능력(공영주차장 관리현황의 주차면수) 추가")
pbl_prk = pd.read_excel(data_path + "공영주차장 관리 현황(영도구)_좌표추가.xlsx", engine="openpyxl")
geom = gpd.points_from_xy(pbl_prk['X'], pbl_prk['Y'], crs=4326)
pbl_prk = gpd.GeoDataFrame(pbl_prk, geometry=geom).to_crs(5179)
pbl_prk['X'] = pbl_prk.geometry.x.round(3)
pbl_prk['Y'] = pbl_prk.geometry.y.round(3)
overlay2 = gpd.overlay(grid, pbl_prk, how="intersection", keep_geom_type=False)
mrg_gdf = mrg_gdf.merge(overlay2, on="gid", how="left")
mrg_gdf.rename(columns={'주차면수': 'prkCnt'}, inplace=True)
mrg_gdf['prkCnt'] = mrg_gdf['prkCnt'].fillna(0)
mrg_gdf = mrg_gdf[['gid', 'CtlCnt', 'fine_amount', 'prkCnt', 'wdr_rd_cd']]
print()
print("도로밀도 추가(기준격자 면적 대비 기준격자 내 도로명DB 실폭도로의 면적 비율)")
sprd_rw = gpd.read_file(publ_path + "TL_SPRD_RW.shp", encoding="cp949").to_crs(5179)
sprd_rw_26200 = sprd_rw.loc[sprd_rw['SIG_CD']=='26200'].copy()
grid['area'] = grid.geometry.area.round(3)
overlay3 = gpd.overlay(grid, sprd_rw_26200, "intersection", keep_geom_type=True)
overlay3['intersect_area'] = overlay3.geometry.area.round(3)
counted3 = overlay3.groupby(['gid','area'])['intersect_area'].sum().reset_index()
mrg_gdf = mrg_gdf.merge(counted3, on="gid", how="left")
mrg_gdf['intersect_area'] = mrg_gdf['intersect_area'].fillna(0)
mrg_gdf['density'] = (mrg_gdf['intersect_area'] / mrg_gdf['area']).round(2)
mrg_gdf = mrg_gdf[['gid', 'CtlCnt', 'fine_amount', 'prkCnt', 'density', 'wdr_rd_cd']]
print()
print("대중교통 접근성 추가(영도구에는 지하철역이 없음 100m 격자 내 버스정류장 집계결과를 활용)")
station = gpd.read_file(publ_path + "tl_bus_station_info.shp", encoding="utf-8").to_crs(5179)
overlay4 = gpd.overlay(grid, station, how="intersection", keep_geom_type=False)
counted4 = overlay4.groupby('gid')[['bstopid']].count().reset_index()
mrg_gdf = mrg_gdf.merge(counted4, on="gid", how="left")
mrg_gdf['bstopid'] = mrg_gdf['bstopid'].fillna(0)
mrg_gdf.rename(columns={'bstopid': 'bsCnt'}, inplace=True)
mrg_gdf = mrg_gdf[['gid', 'CtlCnt', 'fine_amount', 'prkCnt', 'density', 'bsCnt', 'wdr_rd_cd']]
print()
print("상업시설 수 추가: 일반, 집합건축물대장 현황(영도구청 제공)에서 주용도코드명이 상업시설만 추출")
#buld1 = pd.read_excel(data_path + "일반건축물대장 현황.xlsx", engine="openpyxl")
#buld2 = pd.read_excel(data_path + "집합건축물대장 현황.xlsx", engine="openpyxl")
#buld = pd.concat([buld1, buld2], ignore_index=True)
#buld['주용도코드명'] = buld['주용도코드명'].fillna('기타시설')
#buld_filter = buld.loc[(buld['주용도코드명']!='단독주택') & (buld['주용도코드명']!='제1종근린생활시설') & 
#                       (buld['주용도코드명']!='제2종근린생활시설') & (buld['주용도코드명']!='종교시설') &
#                       (buld['주용도코드명']!='국방,군사시설') & (buld['주용도코드명']!='교정및군사시설') &
#                       (buld['주용도코드명']!='발전시설') & (buld['주용도코드명']!='교육연구및복지시설') &
#                       (buld['주용도코드명']!='방송통신시설') & (buld['주용도코드명']!='공동주택')].copy()
#buld_filter.to_csv(publ_path + "건축물대장_상업시설_현황.csv", encoding="utf-8", index=False)
print("상업시설 수 추가: 추출결과에 좌표추가 후 데이터프레임으로 재 변환 실시")
buld = pd.read_excel(data_path + "건축물대장_상업시설_현황_좌표추가.xlsx", engine="openpyxl")
geom = gpd.points_from_xy(buld.경도, buld.위도, crs=4326)
buld_pt = gpd.GeoDataFrame(buld, geometry=geom).to_crs(5179)
overlay5 = gpd.overlay(grid, buld_pt, how="intersection", keep_geom_type=False)
counted5 = overlay5.groupby('gid')[['주용도코드명']].count().reset_index()
mrg_gdf = mrg_gdf.merge(counted5, on="gid", how="left")
mrg_gdf.rename(columns={'주용도코드명': 'CmlrCnt'}, inplace=True)
mrg_gdf = mrg_gdf[['gid', 'CtlCnt', 'fine_amount', 'prkCnt', 'density', 'bsCnt', 'CmlrCnt', 'wdr_rd_cd']]
print()
print("야간상주인구, 직장+방문인구 집계결과(Brightics AI의 WorkFlow Modeler로 작업) 추가")
lifepop = pd.read_csv(pupl_path + "50m격자기준_야간상주_직장방문인구_집계.csv", dtype={'id': str}, encoding="utf-8")
lifepop50 = grid50.merge(lifepop, on='id', how='left')
lifepop50 = lifepop50.loc[~(lifepop50['night_residential'].isnull())].copy()
lifepop50['X'] = lifepop50.centroid.x.round(3)
lifepop50['Y'] = lifepop50.centroid.y.round(3)
geom = gpd.points_from_xy(lifepop50['X'], lifepop50['Y'], crs=5179)
pop50_gdf = gpd.GeoDataFrame(lifepop50, geometry=geom)
overlay6 = grid.overlay(pop50_gdf, how="intersection", keep_geom_type=False)
counted6 = overlay6.groupby('gid')[['night_residential', 'work_visitor_pop']].sum().reset_index()
mrg_gdf = mrg_gdf.merge(counted6, on="gid", how="left")
mrg_gdf = mrg_gdf[['gid', 'CtlCnt', 'fine_amount', 'prkCnt', 'CmlrCnt', 'night_residential',
                   'work_visitor_pop', 'bsCnt', 'density', 'wdr_rd_cd']]
mrg_gdf['CmlrCnt'] = mrg_gdf['CmlrCnt'].fillna(0)
mrg_gdf['density'] = mrg_gdf['density'].fillna(0)
mrg_gdf['wdr_rd_cd'] = mrg_gdf['wdr_rd_cd'].fillna(0)
mrg_gdf['fine_amount'] = mrg_gdf['fine_amount'].fillna(0)
mrg_gdf['night_residential'] = mrg_gdf['night_residential'].fillna(0)
mrg_gdf['work_visitor_pop'] = mrg_gdf['work_visitor_pop'].fillna(0)
print()
print("불법주정차 관리지역 지정용 컬럼 추출 → 타겟변수 정의")
illegal_mng = mrg_gdf[['gid','CtlCnt','prkCnt','CmlrCnt','night_residential',
                       'work_visitor_pop','bsCnt','density','wdr_rd_cd', 'fine_amount']]
illegal_mng.to_csv(out_path + "ml_illegal_analysis_dataset.csv", encoding="utf-8", index=False)
print()
print("공영주차장 건립위치 선정용 컬럼 추출 → 주차수요(타겟변수 정의)")
public_park = mrg_gdf[['gid','prkCnt','CmlrCnt','night_residential','work_visitor_pop','density','bsCnt','wdr_rd_cd']]
cars = cars.loc[(~cars['위도'].isna()) & (~cars['경도'].isna())]
geom = gpd.points_from_xy(cars['경도'], cars['위도'], crs=4326)
cars_pt = gpd.GeoDataFrame(cars, geometry=geom).to_crs(5179)
overlay7 = grid.overlay(cars_pt, how="intersection", keep_geom_type=False)
counted7 = overlay7.groupby('gid')[['차량등록번호']].count().reset_index()
public_park = public_park.merge(counted7, on="gid", how="left")
public_park.rename(columns={'차량등록번호': 'vehicle_count'}, inplace=True)
public_park['vehicle_rate'] = public_park['vehicle_count'] / cars['차량등록번호'].count()
print("회귀모델 구현을 위한 주거지와 상업지별 가중치 계산: 도로밀도에 따라 r값 설정, 상업시설 수에 따라 w값 설정")
r = public_park["density"].apply(lambda x: 0.8 if x > 1.0 else 1.0)  # 도로밀도에 따라 r 값 설정
w = public_park["CmlrCnt"].apply(lambda x: 0.6 if x > 20 else 0.3)  # 상업시설 수에 따라 w 값 설정
public_park['parking_demand'] = (public_park['night_residential'] * r * public_park['vehicle_rate'] + 
                                 public_park['work_visitor_pop'] * w)
public_park['gap_demand_supply'] = public_park['parking_demand'] - public_park['prkCnt']
public_park.to_csv(out_path + "ml_publicpark_dataset.csv", encoding="utf-8", index=False)

불법주정차 과태료 부과건수에 공간속성 부여하기: 차량번호 기준 51,627개 중 14,939개 미부여, 36,688개 부여

도로명주소 시군구 경계에서 영도구를 선택하고 도로구간을 추출

불법주정차 과태료 부과 건수에 도로타입 적용 → 도로명DB 도로구간의 광역도로구분코드 적용
불법주정차 과태료 부과 건수 → 불법주정차 발생을 의미하므로, 발생통계 집계가능, 이를 국토통계격자에 추가

기존 공영주차장 수용능력(공영주차장 관리현황의 주차면수) 추가

도로밀도 추가(기준격자 면적 대비 기준격자 내 도로명DB 실폭도로의 면적 비율)

대중교통 접근성 추가(영도구에는 지하철역이 없음 100m 격자 내 버스정류장 집계결과를 활용)

상업시설 수 추가: 일반, 집합건축물대장 현황(영도구청 제공)에서 주용도코드명이 상업시설만 추출
상업시설 수 추가: 추출결과에 좌표추가 후 데이터프레임으로 재 변환 실시

야간상주인구, 직장+방문인구 집계결과(Brightics AI의 WorkFlow Modeler로 작업) 추가

불법주정차 관리지역 지정용 컬럼 추출 → 타겟변수 정의

공영주차장 건립위치 선정용 컬럼 추출 → 주차수요(타겟변수 정의)
회귀모델 구현을 위한 주거지와 상업지별 가중치 계산


### 불법주정차 관리지역 분류모델 생성
* Random Forest Classifier, XGBoost Classifier

In [151]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import folium
import warnings
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, recall_score, classification_report

warnings.filterwarnings('ignore')

print("1-1. 데이터 준비 및 전처리: 주요 특성과 타겟 변수 선택")
features = ["CtlCnt", "prkCnt", "CmlrCnt", "night_residential", "work_visitor_pop", "bsCnt", "density", "wdr_rd_cd"]
X = illegal_mng[features]
print()
print("1-2. 관리 필요 지역 라벨링 (상위 20% 단속 건수를 기준으로)")
threshold = illegal_mng["CtlCnt"].quantile(0.8)
illegal_mng["target"] = (illegal_mng["CtlCnt"] >= threshold).astype(int)
y = illegal_mng["target"]
print()
print("1-3. 결측값 처리: 결측값은 사전에 처리하여 제외")
#X.fillna(X.mean(), inplace=True)
print()
print("1-4. 학습/테스트 데이터 분리")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print()
print("2-1. 머신러닝 모델 학습")
model = RandomForestClassifier(random_state=42, n_estimators=100)
model.fit(X_train, y_train)
print("2-2. 모델 식 출력 (특성 중요도 확인)")
feature_importances = pd.DataFrame({
    'Feature': features, 'Importance': model.feature_importances_}).sort_values(by='Importance', ascending=False)
print(f"Feature Importances: \n{feature_importances}")
print()
print("2-3. 결정 트리 모델을 통한 규칙 출력")
simple_tree_model = DecisionTreeClassifier(random_state=42, max_depth=3)
simple_tree_model.fit(X_train, y_train)
tree_rules = export_text(simple_tree_model, feature_names=features)
print(f"Decision Tree Rules: \n{tree_rules}")
print()
print("3. 예측")
y_pred = model.predict(X_test)
print()
print("4. 평가")
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print(f"Recall: {recall:.2f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))
print()
print("5. 전체 데이터에 대해 예측값 생성")
illegal_mng["predicted_target"] = model.predict(X)
print()
print("6. 데이터 병합: 격자 데이터와 머신러닝 결과를 gid 기준으로 병합")
merged_gdf = grid.merge(illegal_mng[["gid", "predicted_target"]], on="gid", how="left")
merged_gdf.to_file(oupu_path + "불법주정차 관리지역_분석결과.shp")
print()
print("7. 정적 지도 시각화: 관리 필요 지역(타겟 1)과 불필요 지역(타겟 0)을 다른 색으로 표시")
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
merged_gdf.plot(column="predicted_target", cmap="coolwarm", legend=True, ax=ax)
plt.title("불법 주정차 관리 필요 지역")
legend_labels = {0: "불필요", 1: "필요"}
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=10, label=legend_labels[0]),
           plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, label=legend_labels[1])]
ax.legend(handles=handles, title="범례")
plt.show()
print()
print("8. 인터랙티브 지도 생성: Folium을 사용하여 관리 필요 지역을 인터랙티브하게 시각화")
merged_gdf = merged_gdf.to_crs(epsg=4326)  # Folium은 EPSG:4326 (WGS84) 좌표계 사용
print()
print("9. Folium 지도 생성: 중심좌표(영도구) 설정")
m = folium.Map(location=[35.078650, 129.0648233], zoom_start=12) 
print()
print("10. 격자 데이터를 GeoJson으로 추가: 범례는 HTML로 구현")
folium.GeoJson(
    merged_gdf, name="관리 필요 지역",
    style_function=lambda x: {
        "fillColor": "red" if x["properties"]["predicted_target"] == 1 else "blue",
        "color": "black", "weight": 0.5, "fillOpacity": 0.6}
    ).add_to(m)
legend_html = '''
<div style="position: fixed; 
            bottom: 50px; left: 50px; width: 200px; height: 90px; 
            background-color: white; z-index:9999; font-size:14px; 
            border:2px solid grey; padding: 10px;">
    <b>범례</b><br>
    <i style="background: red; width: 10px; height: 10px; float: left; margin-right: 5px;"></i> 관리 필요<br>
    <i style="background: blue; width: 10px; height: 10px; float: left; margin-right: 5px;"></i> 관리 불필요<br>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))
print()
print("11. 지도 저장: 결과를 HTML 파일로 저장")
m.save(oupu_path + "illegal_parking_map.html")

1-1. 데이터 준비 및 전처리: 주요 특성과 타겟 변수 선택

1-2. 관리 필요 지역 라벨링 (상위 20% 단속 건수를 기준으로)

1-3. 결측값 처리: 결측값은 사전에 처리하여 제외

1-4. 학습/테스트 데이터 분리

2-1. 머신러닝 모델 학습
2-2. 모델 식 출력 (특성 중요도 확인)
Feature Importances: 
             Feature  Importance
0             CtlCnt    0.501178
7          wdr_rd_cd    0.258318
4   work_visitor_pop    0.069688
3  night_residential    0.062264
6            density    0.053401
2            CmlrCnt    0.045854
5              bsCnt    0.008645
1             prkCnt    0.000652

2-3. 결정 트리 모델을 통한 규칙 출력
Decision Tree Rules: 
|--- CtlCnt <= 1.50
|   |--- class: 0
|--- CtlCnt >  1.50
|   |--- class: 1


3. 예측

4. 평가
Accuracy: 1.00
Recall: 1.00
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       393
           1       1.00      1.00      1.00       106

    accuracy                           1.00       499
   macro avg       1.00      1.00      1.00       499
weighted avg       1.00      1.00      1.00       49

### 공영주차장 건립위치 예측모델 생성
* RandomForest Regressor, XGBoost Regressor

In [153]:
import matplotlib.pyplot as plt
import folium
import warnings
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.metrics import mean_squared_error, r2_score

warnings.filterwarnings('ignore')

print("1-1. 데이터 준비 및 전처리: 주요 특성과 타겟 변수 선택 → 주차 수요-공급 격차를 타겟 변수로 사용")
features = ["prkCnt", "CmlrCnt", "night_residential", "work_visitor_pop", "bsCnt", "density", "wdr_rd_cd"]
X = public_park[features]
y = public_park["gap_demand_supply"]
print()
print("1-2. 결측값 처리: X는 주석, y는 수행과정에서 결측값이 발견되어 처리함")
#X.fillna(X.mean(), inplace=True)
y.fillna(y.mean(), inplace=True)
print()
print("1-3. 학습/테스트 데이터 분리")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print()
print("2-1. 머신러닝 모델 학습")
model = RandomForestRegressor(random_state=42, n_estimators=100)
model.fit(X_train, y_train)
print()
print("2-2. 모델 식 출력 (특성 중요도 확인)")
feature_importances = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)
print(f"Feature Importances: \n{feature_importances}")
print()
print("2-3. 결정 트리 모델을 통한 규칙 출력")
simple_tree_model = DecisionTreeRegressor(random_state=42, max_depth=3)
simple_tree_model.fit(X_train, y_train)
tree_rules = export_text(simple_tree_model, feature_names=features)
print(f"Decision Tree Rules: \n{tree_rules}")
print()
print("3. 예측")
y_pred = model.predict(X_test)
print()
print("4. 평가")
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Mean Squared Error: {mse:.2f}")
print(f"R^2 Score: {r2:.2f}")
print()
print("5. 전체 데이터에 대해 예측값 생성")
public_park["predicted_gap"] = model.predict(X)
print()
print("6. 데이터 병합: 격자 데이터와 머신러닝 결과를 gid 기준으로 병합")
merged_gdf = grid.merge(public_park[["gid", "predicted_gap"]], on="gid", how="left")
merged_gdf.to_file(oupu_path + "공영주차장 건립위치_분석결과.shp")
print()
print("6. 정적 지도 시각화: 주차 수요-공급 격차 예측값을 색상으로 표시")
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
merged_gdf.plot(column="predicted_gap", cmap="coolwarm", legend=True, ax=ax)
plt.title("공영주차장 건립 필요 지역 예측")
plt.colorbar(ax.collections[0], ax=ax, label="예측 주차 수요-공급 격차")
plt.show()
print()
print("7. 인터랙티브 지도 생성: Folium을 사용하여 주차 수요-공급 격차를 인터랙티브하게 시각화")
merged_gdf = merged_gdf.to_crs(epsg=4326)
print()
print("8. Folium 지도 생성: 중심좌표(영도구) 설정")
m = folium.Map(location=[35.078650, 129.0648233], zoom_start=12)
print()
print("9. 격자 데이터를 GeoJson으로 추가: : 범례는 HTML로 구현")
folium.GeoJson(
    merged_gdf, name="공영주차장 건립 필요 지역",
    style_function=lambda x: {
        "fillColor": "#ff0000" if x["properties"]["predicted_gap"] > 50 else "#0000ff",
        "color": "black", "weight": 0.5, "fillOpacity": 0.6},
    ).add_to(m)
legend_html = '''
<div style="position: fixed; 
            bottom: 50px; left: 50px; width: 200px; height: 90px; 
            background-color: white; z-index:9999; font-size:14px; 
            border:2px solid grey; padding: 10px;">
    <b>범례</b><br>
    <i style="background: red; width: 10px; height: 10px; float: left; margin-right: 5px;"></i> 검토필요<br>
    <i style="background: blue; width: 10px; height: 10px; float: left; margin-right: 5px;"></i> 검토불필요<br>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))
print()
print("10. 지도 저장: 결과를 HTML 파일로 저장")
m.save(oupu_path + "public_parking_map.html")


1-1. 데이터 준비 및 전처리: 주요 특성과 타겟 변수 선택 → 주차 수요-공급 격차를 타겟 변수로 사용

1-2. 결측값 처리: X는 주석, y는 수행과정에서 결측값이 발견되어 처리함

1-3. 학습/테스트 데이터 분리

2-1. 머신러닝 모델 학습

2-2. 모델 식 출력 (특성 중요도 확인)
Feature Importances: 
             Feature  Importance
3   work_visitor_pop    0.642648
0             prkCnt    0.309238
5            density    0.020410
2  night_residential    0.014622
1            CmlrCnt    0.008952
4              bsCnt    0.002300
6          wdr_rd_cd    0.001830

2-3. 결정 트리 모델을 통한 규칙 출력
Decision Tree Rules: 
|--- work_visitor_pop <= 225.09
|   |--- prkCnt <= 58.00
|   |   |--- work_visitor_pop <= 71.38
|   |   |   |--- value: [9.42]
|   |   |--- work_visitor_pop >  71.38
|   |   |   |--- value: [31.40]
|   |--- prkCnt >  58.00
|   |   |--- prkCnt <= 177.00
|   |   |   |--- value: [-62.61]
|   |   |--- prkCnt >  177.00
|   |   |   |--- value: [-202.17]
|--- work_visitor_pop >  225.09
|   |--- work_visitor_pop <= 385.15
|   |   |--- prkCnt <= 23.00
|   |   |   |--- value: [87.28]
|   |   |--- prkCnt 